In [101]:
import pandas as pd
import numpy as np
import re


def clean_financial_table(df):

    #Checks if a column is a period column. 
    def is_period_column(col):
        if not isinstance(col, str):
            return False
            
        patterns = [
            r"FY\d{2,4}",      # FY2024, FY24
            r"\b\d{4}\b",      # 2024
            r"[A-Za-z]{3,9}\s?\d{2,4}",  # Jun 2024
            r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*",
        ]
        
        return any(re.search(p, col, re.IGNORECASE) for p in patterns)


    def extract_period_unit(col):
        col_clean = col.replace("_", " ") # Temporarily replace underscores with spaces for easier matching
        
        # Pattern to grab "30 June 2025" or "June 2025"
        date_pattern = r"(\d{1,2}\s+)?(january|february|march|april|may|june|july|august|september|october|november|december|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\s+\d{2,4}"
        
        period_match = re.search(date_pattern, col_clean, re.IGNORECASE)
        # If no full date, fall back to FY or Year
        if not period_match:
            period_match = re.search(r"(fy\d{2,4}|\b\d{4}\b)", col_clean, re.IGNORECASE)
            
        # unit_match = re.search(r"\$m|%", col_clean) # Specifically looking for $m or % based on your data
        
        unit_match = re.search(r"\$'?[\w%]*|%+|bps", col_clean)

        period = period_match.group(0) if period_match else col
        unit = unit_match.group(0) if unit_match else None
        
        return period, unit

    # 7️⃣ Clean metric labels (remove units)
    def clean_metric(text):

        if pd.isna(text):
            return text

        text = str(text).lower()

        # remove units like ($m)
        text = re.sub(r"\(.*?\)", "", text)

        # remove trailing footnote numbers
        text = re.sub(r"\s+\d+(,?\d)*", "", text)

        # remove superscript footnotes
        text = re.sub(r"[¹²³⁴⁵⁶⁷⁸⁹⁰]", "", text)

        # remove star footnotes
        text = re.sub(r"\*", "", text)

        text = text.strip()

        return text

    # 8️⃣ Cleans datapoints to ensure that there are no footnotes etc. 
    def clean_number(x):

        if pd.isna(x):
            return x

        x = str(x).strip()

        # detect negative via parentheses
        is_negative = bool(re.search(r"\(\s*\d+\.?\d*", x))

        # remove commas
        x = x.replace(",", "")

        # remove footnote markers
        x = re.sub(r"[¹²³⁴⁵⁶⁷⁸⁹⁰\*]", "", x)

        # extract all numbers
        numbers = re.findall(r"\d+\.?\d*", x)

        if numbers:
            value = float(numbers[0])

            if is_negative:
                value = -value

            # percentage handling
            if "%" in x:
                return value / 100

            return value

        return x 


    df = df.copy()

    # 1️⃣ Replace blank strings with NaN
    df = df.replace(r'^\s*$', np.nan, regex=True)

    # 2️⃣ Drop completely empty rows and columns
    df = df.dropna(axis=0, how="all")
    df = df.dropna(axis=1, how="all")

    # 3️⃣ Reset index
    df = df.reset_index(drop=True)

    # 4️⃣ Detect header row
    header_row = None

    for i in range(min(3, len(df))):

        row_text = " ".join(df.iloc[i].astype(str).str.lower())

        # header_patterns = r"(?i)\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|fy|hy|q[1-4]|20\d{2})\b"
        header_patterns = r"(?i)\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|january|february|march|april|may|june|july|august|september|october|november|december)\b|fy|hy|q[1-4]|20\d{2}"
        # r'(fy|hy|FY|HY|20\d{2}|q[1-4])'
        if re.search(header_patterns, row_text):
            header_row = i
            break

    # Promote header if detected
    if header_row is not None:

        df.columns = df.iloc[header_row]
        df = df.drop(header_row).reset_index(drop=True)

    else:
        df.columns = [f"col_{i}" for i in range(len(df.columns))]

    # 5️⃣ Normalize column names
    df.columns = (
        pd.Series(df.columns)
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )

    # 6️⃣ Ensure first column is metric column
    df = df.rename(columns={df.columns[0]: "metric"})

    
    print(df.columns)
    #This removes unnecessary columns 
    period_cols = [c for c in df.columns if is_period_column(str(c))]
    df = df[["metric"] + period_cols]
    



    #Code here stores units and period into separate dictionaries. 
    period_map = {}
    unit_map = {}

    for col in period_cols:
        period, unit = extract_period_unit(col)
        period_map[col] = period
        
        
        unit_map[col] = unit
        

    
    #Cleans only for metric column.
    df['metric'] = df['metric'].map(clean_metric)

    #Apply clean_number_conditionally only to non metric cols. 
    cols_excluding_metric = df.columns.difference(['metric'])
    df[cols_excluding_metric] = df[cols_excluding_metric].map(clean_number)

    df_long = df.melt(
        id_vars = "metric",
        var_name = "column",
        value_name = "value"
    )

    df_long["period"] = df_long["column"].map(period_map)
    df_long["unit"] = df_long["column"].map(unit_map)

    return df_long

In [102]:
test_table = r"datasets\processed\tables\GMG\FY24\GMG_FY24_IP-table-0"
# test_table = r"datasets\processed\tables\ABG\FY17\ABG_FY17_IP-table-5"

df = pd.read_csv(test_table,index_col=0,header=None)

In [103]:
df

,1,2,3
0,,,
NaN,NaN,FY23,FY24
0.0,Operatingprofit($M),"1,783.2","2,049.4"
1.0,Statutory accountingprofit / (loss) ($M),"1,559.9",(98.9)
2.0,OperatingEPS(cents)¹,94.3,107.5
3.0,Distributionper security (cents),30.0,30.0
4.0,NaN,ASAT 30JUNE2023,ASAT 30JUNE2024
5.0,NTApersecurity($),9.12,8.80
6.0,Gearing (balance sheet) (%)²,8.3,8.4
7.0,Availableliquidity ($B),3.1,3.8


In [104]:
clean_financial_table(df=df)

Index(['metric', 'fy23', 'fy24'], dtype='object', name=0)


,metric,column,value,period,unit
0,operatingprofit,fy23,1783.20,fy23,None
1,statutory accountingprofit /,fy23,1559.90,fy23,None
2,operatingeps,fy23,94.30,fy23,None
3,distributionper security,fy23,30.00,fy23,None
4,NaN,fy23,30.00,fy23,None
5,ntapersecurity,fy23,9.12,fy23,None
6,gearing,fy23,8.30,fy23,None
7,availableliquidity,fy23,3.10,fy23,None
8,wacr,fy23,4.50,fy23,None
9,operatingprofit,fy24,2049.40,fy24,None
